# LILaC reproduction (MMCoQA) — Colab driver
Runs `scripts/lilac/pipeline.sh` stage by stage. Each GPU stage is resumable; run `sync.sh push` after every stage.

### 0. Session setup
Runtime → *Change runtime type* → **T4 GPU** (free tier) or L4 / A100 (Colab Pro). On a T4 the notebook uses 4-bit AWQ checkpoints for Qwen2.5-VL and the decomposers and loads MM-Embed with 8-bit weights (`MMEMBED_QUANT=8bit`); keep the same setting for all MM-Embed stages. Free-tier sessions can be cut at any time: every stage is resumable, so after reconnecting run cells 0–2 again and continue (step 2 restores artefacts from Drive).

Store `WANDB_API_KEY` and `HF_TOKEN` in Colab *Secrets* (key icon on the left) and enable *Notebook access* for both. Never paste keys into the notebook: the repository is public.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
for k in ('WANDB_API_KEY', 'HF_TOKEN'):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print('secret not set:', k)
# W&B runs go to the default entity of WANDB_API_KEY; set WANDB_ENTITY to use a team instead.
os.environ['ENV_ROOT'] = '/content/envs'

import codecs, signal, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command and stream its output. A failing command raises an error,
    so 'Run all' stops at the first failed stage instead of running the next ones."""
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, start_new_session=True)
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    try:
        while True:
            b = os.read(p.stdout.fileno(), 4096)
            if not b:
                break
            sys.stdout.write(dec.decode(b)); sys.stdout.flush()
        rc = p.wait()
    except KeyboardInterrupt:          # 'Interrupt execution' also stops the GPU job
        os.killpg(p.pid, signal.SIGINT)
        try:
            p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        raise
    if rc != 0 and check:
        raise RuntimeError(f'command failed (exit code {rc}): {cmd}')

In [ ]:
%cd /content
sh('git clone https://github.com/NoiceSmoker/mmgraphrag-repro.git 2>/dev/null || (cd mmgraphrag-repro && git pull)')
%cd /content/mmgraphrag-repro
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
T4 = 'T4' in gpu
if T4:   # 16 GB, no bf16 (vLLM switches to fp16 by itself)
    os.environ['VLM_MODEL'] = 'Qwen/Qwen2.5-VL-7B-Instruct-AWQ'
    os.environ['DECOMPOSERS'] = ('qwen2.5-7b-awq=Qwen/Qwen2.5-7B-Instruct-AWQ '
                                 'qwen2.5-14b-awq=Qwen/Qwen2.5-14B-Instruct-AWQ')
    os.environ['MMEMBED_QUANT'] = '8bit'          # MM-Embed in LLM.int8 (fp16 does not fit 16 GB)
    os.environ['MMEMBED_BATCH_SCALE'] = '0.5'     # smaller batches; halved again automatically on OOM
print(gpu, '| T4 mode' if T4 else '')

### 1. Environments (once per VM, ~5–10 min)

In [ ]:
sh('bash scripts/setup_envs.sh lilac-base mmembed vllm')

### 2. Official code + data (restores earlier artefacts from Drive if present)

In [ ]:
sh('bash scripts/colab/sync.sh pull')
sh('bash scripts/lilac/pipeline.sh fetch data')

### 3. Graph construction: Qwen2.5-VL passes (vLLM) + official steps 1–7

In [ ]:
sh('bash scripts/lilac/pipeline.sh vlm && bash scripts/colab/sync.sh push')     # long GPU job, saved right after
sh('bash scripts/lilac/pipeline.sh graph && bash scripts/colab/sync.sh push')

### 4. MM-Embed corpus embeddings (8-bit on a T4; resumable per component)

In [ ]:
sh('bash scripts/lilac/pipeline.sh embed && bash scripts/colab/sync.sh push')

### 5. Query decomposition with several LLMs + sub-query embeddings
`decompose_llm` writes the sub-queries, `decompose_embed` embeds them with the same MM-Embed setting as step 4. On an L4 drop the 32B model: `DECOMPOSERS='qwen2.5-7b=Qwen/Qwen2.5-7B-Instruct qwen2.5-14b-awq=Qwen/Qwen2.5-14B-Instruct-AWQ'`.

In [ ]:
sh('bash scripts/lilac/pipeline.sh decompose_llm && bash scripts/colab/sync.sh push')
sh('bash scripts/lilac/pipeline.sh decompose_embed && bash scripts/colab/sync.sh push')

### 6. (optional) extra ablation: image nodes without VLM text

In [ ]:
sh('bash scripts/lilac/pipeline.sh novlm && bash scripts/colab/sync.sh push')

### 7. Retrieval experiments → W&B (CPU is enough; can also run locally)

In [ ]:
sh('bash scripts/lilac/pipeline.sh experiments && bash scripts/colab/sync.sh push')

### 8. W&B report pages + LaTeX tables for the written report

In [ ]:
sh('/content/envs/lilac-base/bin/python scripts/make_wandb_reports.py --share')
sh('/content/envs/lilac-base/bin/python scripts/export_results.py --results results --out results/latex')